In [1]:
import sys
import os

# Project root (adjust if notebook location differs)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root added: {project_root}")

Project root added: c:\Users\Shiwan\OneDrive\Desktop\multi_agent_customer_support_system\layer_2_triage


In [2]:
import json
import pandas as pd
from datetime import datetime
import logging

from graphs.state_factory import TriageStateFactory
from graphs.triage_graph import build_triage_graph

logging.basicConfig(level=logging.INFO)

c:\Users\Shiwan\OneDrive\Desktop\multi_agent_customer_support_system\venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
sample_ticket = {
    "ticket_id": "TKT-2002",
    "customer_email": "john@example.com",
    "intent": "refund_request",
    "urgency": "medium",
    "sentiment": "neutral",
    "confidence": 0.95,
    "channel": "email",
    "message_raw": "I want a refund.",
    "message_english": "I want a refund.",
    "entities": {
        "order_id": 14
    }
}
print(f"Input payload prepared: {sample_ticket['ticket_id']}")

Input payload prepared: TKT-2002


In [4]:
initial_state = TriageStateFactory.create_triage_state(sample_ticket)

print("Initial state created.")
print(f"State keys: {list(initial_state.keys())}")

Initial state created.
State keys: ['ticket', 'entities', 'ticket_id', 'customer_email', 'customer_id', 'initial_intent', 'initial_urgency', 'initial_sentiment', 'supervisor_confidence', 'customer_tier', 'ltv', 'unresolved_repeat_count', 'total_tickets', 'total_escalations', 'last_sentiment', 'order_context', 'urgency_score', 'ltv_score', 'sentiment_score', 'history_score', 'final_score', 'final_priority', 'sla_duration_hours', 'sla_deadline', 'insight_tags', 'escalation_required', 'escalation_reason', 'created_at', 'next_agent', 'current_node', 'workflow_logs']


In [5]:
triage_app = build_triage_graph()

print("Triage graph compiled successfully.")

Triage graph compiled successfully.


In [6]:
print("Running triage workflow...")
config = {"configurable": {"thread_id":"2"}}
final_state = triage_app.invoke(initial_state,config=config)

print("Workflow execution complete.")

Running triage workflow...
Workflow execution complete.


In [7]:
results_summary = {
    "Priority": final_state.get("final_priority"),
    "SLA (Hours)": final_state.get("sla_duration_hours"),
    "Final Score": final_state.get("final_score"),
    "Dispatch Target": final_state.get("next_agent"),
    "Escalation Flag": final_state.get("escalation_required"),
    "Escalation Reason": final_state.get("escalation_reason")
}

for key, value in results_summary.items():
    print(f"{key:<20}: {value}")

Priority            : LOW
SLA (Hours)         : 48
Final Score         : 3.5
Dispatch Target     : refund_agent
Escalation Flag     : False
Escalation Reason   : None


In [8]:
logs = final_state.get("workflow_logs", [])

df_logs = pd.DataFrame(logs)

if not df_logs.empty:
    df_logs["timestamp"] = pd.to_datetime(df_logs["timestamp"]).dt.strftime("%H:%M:%S")
    display(df_logs[["timestamp", "node", "message"]])
else:
    print("No workflow logs found.")

,timestamp,node,message
0,15:48:15,state_factory,Workflow initialized for ticket TKT-2002.
1,15:48:15,fetch_customer_node,CRM identity and LTV successfully loaded.
2,15:48:15,fetch_order_node,Transaction context for 14 loaded.
3,15:48:15,history_node,Historical behavioral signals successfully ana...
4,15:48:15,scoring_node,Deterministic scoring complete. Final Score: 3.5
5,15:48:15,priority_node,Business priority successfully classified as LOW.
6,15:48:15,sla_node,SLA commitment of 48h assigned.
7,15:48:15,escalation_check_node,Global escalation policy evaluated.
8,15:48:15,dispatch_node,Ticket dispatched to refund_agent.


In [9]:
scoring_data = next(
    (log["data"] for log in logs if log["node"] == "scoring_node"),
    None
)

if scoring_data:
    print("Scoring breakdown:")
    print(json.dumps(scoring_data, indent=4))
else:
    print("No scoring data found.")

Scoring breakdown:
{
    "urgency_score": 5.0,
    "ltv_score": 2.01,
    "sentiment_score": 4.5,
    "history_score": 0.0,
    "final_score": 3.5
}


In [10]:
results_summary = {
    "Priority": final_state.get("final_priority"),
    "SLA (Hours)": final_state.get("sla_duration_hours"),
    "Final Score": final_state.get("final_score"),
    "Dispatch Target": final_state.get("next_agent"),
    "Escalation Flag": final_state.get("escalation_required"),
    "Escalation Reason": final_state.get("escalation_reason")
}

for key, value in results_summary.items():
    print(f"{key:<20}: {value}")

Priority            : LOW
SLA (Hours)         : 48
Final Score         : 3.5
Dispatch Target     : refund_agent
Escalation Flag     : False
Escalation Reason   : None


In [11]:
print("Resolved Customer:", final_state["customer_id"])

Resolved Customer: 5


In [12]:
print(final_state["insight_tags"])

['high_value_transaction']
